In [1]:
import os
import sys
dir_path = "/qumulo/shared_data/aofei_summer/RegTok/RegLLM"
sys.path.insert(0, dir_path)
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
os.environ["HF_HUB_CACHE"]="/qumulo/shared_data/aofei_summer/LLMs"
from llava.eval.cli_v1 import RegLLMChatbot

/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/qumulo/shared_data/aofei_summer/miniconda3/lib/python3.13/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# model_dir = "/qumulo/shared_data/aofei_summer/RegTok/RegLLM/checkpoints/i2t_instruct"
# model_dir = "/qumulo/shared_data/aofei_summer/RegTok/RegLLM/checkpoints/i2t_instruct_full"
peft_path = "/qumulo/shared_data/aofei_summer/RegTok/RegLLM/checkpoints/sft_slake"
bot = RegLLMChatbot(model_dir=None, peft_path=peft_path, device="cuda")

loading model from None


You are using a model of type qwen3 to instantiate a model of type llava_qwen. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 5/5 [00:00<00:00, 86.01it/s]


load vision tower!
Number of stacks: 1
Upsample mode: conv
tokenflow load from: /qumulo/shared_data/aofei_summer/RegTok/source/RegTok_pipeline_full_wo_quant/002-RegTok/checkpoints/0079280.pt
tokenflow model load success!!
pre loading complete!
Loading LoRA weights from /qumulo/shared_data/aofei_summer/RegTok/RegLLM/checkpoints/sft_slake
Merging weights
Unexpected peft weights: []


In [3]:
bot.inference("What modality is used to take this image?", images="/qumulo/shared_data/aofei_summer/data/evaluation/imgs/xmlab102/source.jpg")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


['assistant\nCT']

In [5]:
import json
from tqdm import tqdm
model_name = "sft"
question_file = "/qumulo/shared_data/aofei_summer/data/evaluation/test.json"
answers_file = f"/qumulo/shared_data/aofei_summer/data/evaluation/inference/answers_{model_name}.jsonl"
image_folder = "/qumulo/shared_data/aofei_summer/data/evaluation/imgs"

In [8]:
questions = json.load(open(os.path.expanduser(question_file), "r"))
answers_file = os.path.expanduser(answers_file)
os.makedirs(os.path.dirname(answers_file), exist_ok=True)
ans_file = open(answers_file, "w")
for line in tqdm(questions):

    idx = line["qid"]
    question = line["question"] # ['value'].split('\n')[0]
    gt_ans = line["answer"] # ['value']      
    image_file = line["img_name"]

    qs = question
    
    image_file = os.path.join(image_folder, image_file)
    ans = bot.inference(qs, image_file)[0]
    ans = ans.replace("assistant\n", "").strip()

    ans_file.write(json.dumps({"question_id": idx,
                                   "prompt": qs,
                                   "text": ans,
                                   "gt_ans": gt_ans,
                                   "metadata": {}}) + "\n")
    ans_file.flush()
ans_file.close()

  0%|          | 0/2094 [00:00<?, ?it/s]

100%|██████████| 2094/2094 [23:23<00:00,  1.49it/s]
